# Comparing IS aggregation and VTS on retail sales

Virtual Typical Subject (VTS) and Individual Structure (IS) are two ways to
summarise a multi-subject cohort into one group-level structure.

- **VTS** (`compute_vts`) builds one representative multivariate series, then fits a single MDM.
- **IS** (`compute_is`) fits one MDM per subject and aggregates the DAGs.

This notebook uses the bundled retail dataset (`load_retail()`, same data as
`01-mdmp-library-demo.ipynb`). Subjects are calendar months with a
shared set of product-line nodes. There is no ground-truth DAG; we compare the
two group summaries side by side.


## Load retail data and build monthly subjects

`load_retail()` returns the bundled sales panel. We keep one SKU per product line so
node meaning is shared across months (required for VTS/IS).


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from mdmp import (
    MDM,
    SKU_DAG_LABELS,
    compute_is,
    compute_vts,
    fit_individual_structures,
    load_retail,
    monthly_subjects,
    one_sku_per_type,
    plot_dag,
)

sales, hierarchy = load_retail()
monthly_nodes = one_sku_per_type(hierarchy)
node_labels = [SKU_DAG_LABELS.get(s, s) for s in monthly_nodes]
subjects, subject_ids, _ = monthly_subjects(sales, node_names=monthly_nodes)

# Use a short cohort so the notebook runs quickly
subjects = subjects[:6]
subject_ids = subject_ids[:6]
print(f"{len(subjects)} months, N={len(node_labels)} nodes, T~{subjects[0].shape[0]}")
print("subject_ids:", subject_ids)


## Fit one MDM per subject

`fit_individual_structures` is the GS/IS stage-1 helper: hill-climbing MDM on
each monthly panel.


In [ ]:
NBF = 10
individuals = fit_individual_structures(
    subjects,
    method="hc",
    nbf=NBF,
    node_names=node_labels,
    subject_ids=subject_ids,
    n_jobs=-1,
    verbose=False,
)
print([m.adj_mat.sum() for m in individuals], "edges per subject (count)")


## VTS path

Average across subjects (after aligning time), then fit one MDM on the
representative series.


In [ ]:
vts = compute_vts(subjects, method="mean")
vts_df = pd.DataFrame(vts.vts_data, columns=node_labels)
model_vts = MDM(vts_df, method="hc", nbf=NBF, verbose=False, n_jobs=-1)
plot_dag(model_vts, plot_type="graph", title="VTS (mean) DAG")
plt.show()


## IS path

Aggregate the subject MDMs with edge voting (`tau=0.5`). The result exposes the
same plotting interface as a fitted MDM for DAG views.


In [ ]:
is_view = compute_is(
    individuals,
    tau=0.5,
    mc_n_samples=0,
    node_names=node_labels,
)
plot_dag(is_view, plot_type="graph", title="IS consensus DAG (tau=0.5)")
plt.show()
freqs = is_view.metadata.get("edge_frequencies")
if freqs is not None:
    print(pd.DataFrame(freqs, index=node_labels, columns=node_labels).round(2))


## Heatmap comparison

Compare adjacency matrices from VTS and IS. Differences highlight how series
averaging versus structure voting summarise the same cohort.


In [ ]:
plot_dag(model_vts, plot_type="heatmap", title="VTS adjacency")
plt.show()
plot_dag(is_view, plot_type="heatmap", title="IS adjacency")
plt.show()
